# CLeaning Pipeline

## Import Libraries and Loading

In [ ]:
%matplotlib inline

%pip install recordlinkage --quiet

import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.loader import load_tables
from utils.schema_guard import SchemaContract
from utils.dqa import run_dqa
from utils.domain_rules import get_domain_rules
from utils.cleaning import CleaningPipeline
import recordlinkage

os.makedirs(CLEANED_DATA_DIR, exist_ok=True)
os.makedirs(CLEANING_REPORT_DIR, exist_ok=True)

print("Libraries imported and settings configured")

## Loading Reconciled Database

In [ ]:
dfs_raw = load_tables("reconciled", normalize_cols="lower")

# Register schemas for post-cleaning validation
contract = SchemaContract()
for name, df in dfs_raw.items():
    contract.register(name, df)

print(f"\nRegistered {len(contract.registered_tables)} table schemas.")

pre_report = contract.validate_all(dfs_raw)
if pre_report["failures"] == 0:
    print("Pre-cleaning schema validation PASSED - raw tables match registered schemas.")
else:
    print("Pre-cleaning schema validation FAILED - unexpected structural issues detected!")
    for entry in pre_report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")

## Domain Rules

In [ ]:
domain_rules = get_domain_rules()

print("Domain Rules configured (with referential integrity via closures).")
print(f"Tables covered: {list(domain_rules.keys())}")

## Cleaning Rules

In [ ]:
cleaning_rules = {
    "Flights": {
        "pk": "flight_id",
        "uppercase_cols": ["origin", "dest", "op_unique_carrier", "mkt_unique_carrier", "tail_num"],
        "date_cols": {"fl_date": "%Y-%m-%d"},
        "winsorize_cols": ["taxi_out", "air_time"],
        "winsorize_bounds": (0.00, 0.99),
        "fk_checks": [
            {"col": "origin", "ref_table": "Stations", "ref_col": "airport"},
            {"col": "dest", "ref_table": "Stations", "ref_col": "airport"},
            {"col": "tail_num", "ref_table": "Aircrafts", "ref_col": "tail_num"},
            {"col": "mkt_unique_carrier", "ref_table": "Carriers", "ref_col": "code"},
            {"col": "op_unique_carrier", "ref_table": "Carriers", "ref_col": "code"},
            {"col": "cancelled", "ref_table": "Cancellation", "ref_col": "status"},
        ],
        "validity_fixes": [],
    },
    "Weather_Observations": {
        "pk": "obs_id",
        "uppercase_cols": ["origin_airport"],
        "date_cols": {"obs_date": "%Y-%m-%d"},
        "dedup_key": ["origin_airport", "obs_date", "obs_hour"],
        "validity_fixes": [
            {"col": "rel_humidity", "min_val": 0, "max_val": 100},
            {"col": "altimeter", "min_val": 25.0, "max_val": 35.0},
            {"col": "visibility", "min_val": 0.0, "max_val": 20.0},
        ],
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "fk_checks": [
            {"col": "origin_airport", "ref_table": "Stations", "ref_col": "airport"},
            {"col": "active_weather_status", "ref_table": "Active_Weather", "ref_col": "status"},
        ],
    },
    "Aircrafts": {
        "pk": "tail_num",
        "uppercase_cols": ["tail_num", "icao_type"],
        "titlecase_cols": ["manufacturer"],
        "fix_tail_num": True,
        "date_cols": {},
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "validity_fixes_categorical": [
            {"col": "manufacturer", "valid_values": ["Airbus", "Boeing", "Bombardier", "Embraer", "Unknown"]},
            {"col": "icao_type", "valid_values": None, "pattern": r"^([A-Z0-9]{2,4})$"},
        ],
        "fk_checks": [],
    },
    "Carriers": {
        "pk": "code",
        "uppercase_cols": ["code"],
        "titlecase_cols": ["description"],
        "date_cols": {},
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "fk_checks": [],
    },
    "Stations": {
        "pk": "airport",
        "uppercase_cols": ["airport", "airport_state_code", "icao", "iata", "faa", "mesonet_station"],
        "titlecase_cols": ["display_airport_name", "display_airport_city_name_full", "airport_state_name"],
        "date_cols": {},
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "validity_fixes": [
            {"col": "elevation", "min_val": -1500},
        ],
        "fk_checks": [],
    },
    "Cancellation": {
        "pk": "status",
        "titlecase_cols": ["cancellation_reason"],
        "date_cols": {},
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "fk_checks": [],
    },
    "Active_Weather": {
        "pk": "status",
        "date_cols": {},
        "winsorize_cols": [],
        "winsorize_bounds": (0.00, 1.00),
        "fk_checks": [],
    },
}

print("Cleaning rules configured.")

## Pipeline Execution

In [ ]:
all_audit_logs = []
dfs_clean = {}
all_scorecards = []
raw_row_counts = {table_name: len(df) for table_name, df in dfs_raw.items()}
raw_null_counts = {table_name: int(df.isnull().sum().sum()) for table_name, df in dfs_raw.items()}

TABLE_ORDER = [
    "Active_Weather",
    "Cancellation",
    "Carriers",
    "Stations",
    "Aircrafts",
    "Weather_Observations",
    "Flights",
]

for table_name in TABLE_ORDER:
    if table_name not in dfs_raw:
        continue
    df = dfs_raw[table_name]
    print(f"\n{'='*60}\nCLEANING: {table_name.upper()}\n{'='*60}")

    rules = cleaning_rules.get(table_name, {})
    pipeline = CleaningPipeline(df, table_name, pk_col=rules.get("pk"))

    dqa_before = run_dqa(df, table_name, domain_rules, dfs_ref=dfs_raw)
    sc_before = dqa_before.scorecard().copy()
    sc_before.insert(0, "Phase", "Before")

    # String Normalization
    if rules.get("uppercase_cols"):
        pipeline.standardize_strings(rules["uppercase_cols"], upper=True, strip=True)
    if rules.get("titlecase_cols"):
        pipeline.standardize_strings(rules["titlecase_cols"], title_case=True, strip=True)

    # Date Parsing
    for col, fmt in rules.get("date_cols", {}).items():
        pipeline.parse_dates(col, date_format=fmt)

    # Weather_Observations specific
    if table_name == "Weather_Observations":
        pipeline.handle_weather_nulls(
            weather_cols=WEATHER_COLS,
            flight_df=dfs_raw.get("Flights"),
            strategy="auto",
        )
        if rules.get("dedup_key"):
            pipeline.deduplicate(rules["dedup_key"])
        pipeline.reset_serial_id("obs_id")
        for vfix in rules.get("validity_fixes", []):
            pipeline.fix_validity_range(**vfix)
        pipeline.fix_dew_point()
        pipeline.fix_wind_gust()
        pipeline.fix_cloud_cover_consistency()

    # Aircrafts specific
    if table_name == "Aircrafts":
        if rules.get("fix_tail_num"):
            pipeline.flag_invalid_tail_num()
        for cfg in rules.get("validity_fixes_categorical", []):
            col = cfg["col"]
            if cfg.get("valid_values") is not None:
                pipeline.fix_invalid_to_null(col, cfg["valid_values"], reason=f"{col}: value not in allowed set -> NULL")
            elif cfg.get("pattern") and col in pipeline.df.columns:
                before = pipeline.df[col].copy()
                invalid = pipeline.df[col].notna() & ~pipeline.df[col].astype(str).str.match(cfg["pattern"])
                if invalid.sum() > 0:
                    pipeline.df.loc[invalid, col] = np.nan
                    pipeline.audit.log_batch("fix_invalid_pattern", col, invalid, before, pipeline.df[col], f"{col}: pattern mismatch -> NULL")
                    print(f"  [fix_invalid_pattern:{col}] {invalid.sum()} values set to NULL.")

    # Flights specific
    if table_name == "Flights":
        pipeline.fix_cancelled_air_time()

    # Validity range fixes (generic)
    if table_name not in ("Weather_Observations", "Aircrafts"):
        for vfix in rules.get("validity_fixes", []):
            pipeline.fix_validity_range(**vfix)

    # FK integrity
    for fk in rules.get("fk_checks", []):
        ref_df = dfs_clean.get(fk["ref_table"])
        if ref_df is None:
            print(f"  [fk_check] WARNING: {fk['ref_table']} not yet cleaned - skipping FK for {fk['col']}")
            continue
        valid_keys = ref_df[fk["ref_col"]].dropna().unique()
        pipeline.fix_fk_to_null(fk["col"], valid_keys, reason=f"FK -> {fk['ref_table']}.{fk['ref_col']} broken -> NULL")

    # Rebuild weather_observation_id via natural key
    if table_name == "Flights":
        wo_clean = dfs_clean.get("Weather_Observations")
        if wo_clean is not None:
            pipeline.fix_weather_observation_fk(wo_clean)
        else:
            print("  [fix_weather_observation_fk] WARNING: Weather_Observations not yet cleaned - skipping.")

    # Generic Winsorize
    win_cols = rules.get("winsorize_cols", [])
    if win_cols:
        low, high = rules.get("winsorize_bounds", (0.01, 0.99))
        pipeline.winsorize_numeric(win_cols, lower_pct=low, upper_pct=high)

    # Schema validation after cleaning
    validation_result = contract.validate(table_name, pipeline.df, strict=False)
    if validation_result["status"] == "FAIL":
        print("  [schema_validate] FAILED after cleaning:")
        for entry in validation_result["details"]:
            if entry["status"] != "PASS":
                print(f"    {entry}")
    elif validation_result["status"] == "WARN":
        print("  [schema_validate] WARN after cleaning (non-fatal):")
        for entry in validation_result["details"]:
            if entry["status"] == "WARN":
                print(f"    {entry}")

    # Save
    df_clean = pipeline.clean_df
    dfs_clean[table_name] = df_clean
    out_path = CLEANED_PATHS.get(table_name, CLEANED_DATA_DIR / f"{table_name}.csv")
    df_clean.to_csv(out_path, index=False)
    print(f"  Saved -> {out_path}")

    # DQA AFTER (using cleaned dfs for FK checks)
    dqa_after = run_dqa(df_clean, table_name, domain_rules, dfs_ref=dfs_clean)
    sc_after = dqa_after.scorecard().copy()
    sc_after.insert(0, "Phase", "After")

    # Comparison table
    comparison = sc_before.merge(sc_after[["Dimension", "Score", "Issues", "Status", "Phase"]], on="Dimension", suffixes=("_before", "_after"))
    comparison["Delta"] = (comparison["Score_after"] - comparison["Score_before"]).round(4)
    comparison["Improved"] = comparison["Delta"].apply(lambda x: "YES" if x > 0 else ("=" if x == 0 else "NO"))
    print(f"\n  Overall: {dqa_before.overall_score():.4%} -> {dqa_after.overall_score():.4%}  " f"(Delta: {dqa_after.overall_score() - dqa_before.overall_score():+.4%})")
    print(comparison[["Dimension", "Score_before", "Score_after", "Delta", "Status_before", "Status_after", "Improved"]].to_string(index=False))

    # Before/After Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    def _color(score):
        return "#2ecc71" if score >= 0.95 else ("#f1c40f" if score >= 0.80 else "#e74c3c")

    for ax, phase, dqa_obj in [(axes[0], "Before", dqa_before), (axes[1], "After", dqa_after)]:
        sc = dqa_obj.scorecard().sort_values("Score", ascending=True)
        colors = [_color(s) for s in sc["Score"]]
        bars = ax.barh(sc["Dimension"], sc["Score"], color=colors, height=0.6, alpha=0.85, zorder=3)
        ax.set_xlim(0, 1.15)
        ax.axvline(0.95, color="#cccccc", linestyle="--", linewidth=1.2, zorder=0)
        ax.text(0.90, -0.75, "Target (95%)", color="#888888", fontsize=9, ha="left", va="top")
        ax.axvline(0.80, color="#cccccc", linestyle="--", linewidth=1.2, zorder=0)
        ax.text(0.85, -0.75, "Warn (80%)", color="#888888", fontsize=9, ha="right", va="top")
        for bar in bars:
            w = bar.get_width()
            ax.text(w + 0.015, bar.get_y() + bar.get_height() / 2, f"{w:.2%}", va="center", fontsize=10, fontweight="semibold", color="#333333")
        ax.set_title(f"{phase} Cleaning - {table_name.upper()}\nOverall: {dqa_obj.overall_score():.2%}", fontsize=13, fontweight="bold", loc="left", pad=12, color="#333333")
        ax.set_xticks([])
        ax.tick_params(axis="y", length=0, labelsize=10, labelcolor="#555555")
        for spine in ["top", "right", "bottom", "left"]:
            ax.spines[spine].set_visible(False)

    plt.suptitle(f"Data Quality - Before vs After Cleaning: {table_name.upper()}", fontsize=14, fontweight="bold", y=1.02, color="#222222")
    plt.tight_layout()
    plt.savefig(CLEANING_REPORT_DIR / f"Plot_BeforeAfter_{table_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Save scorecards
    sc_before.to_csv(CLEANING_REPORT_DIR / f"Scorecard_Before_{table_name}.csv", index=False)
    sc_after.to_csv(CLEANING_REPORT_DIR / f"Scorecard_After_{table_name}.csv", index=False)
    comparison.to_csv(CLEANING_REPORT_DIR / f"Scorecard_Comparison_{table_name}.csv", index=False)
    all_scorecards.append(comparison)

    # Audit Summary
    summary = pipeline.audit.summary()
    if not summary.empty:
        print("\n  --- Audit Summary ---")
        print(summary.to_string(index=False))
        summary.to_csv(CLEANING_REPORT_DIR / f"Audit_Summary_{table_name}.csv", index=False)

    log_df = pipeline.audit.to_df()
    if not log_df.empty:
        log_df.insert(0, "Table", table_name)
        all_audit_logs.append(log_df)

# Global outputs
if all_audit_logs:
    full_log = pd.concat(all_audit_logs, ignore_index=True)
    full_log.to_csv(CLEANING_REPORT_DIR / "Complete_Cleaning_Audit_Log.csv", index=False)
    print(f"\n{'─'*60}\nComplete Audit Log: {len(full_log):,} transformation records.")

if all_scorecards:
    pd.concat(all_scorecards, ignore_index=True).to_csv(CLEANING_REPORT_DIR / "Global_Scorecard_Comparison.csv", index=False)
    print("Global Scorecard saved.")

## Fuzzy Deduplication (RecordLinkage)

In [ ]:
print("\nRunning fuzzy deduplication on Stations...")

stations = dfs_clean.get("Stations", dfs_raw.get("Stations")).copy().reset_index(drop=True)

indexer = recordlinkage.Index()
if "airport_state_code" in stations.columns:
    indexer.block("airport_state_code")
else:
    indexer.full()

candidate_links = indexer.index(stations)
compare = recordlinkage.Compare()

if "display_airport_city_name_full" in stations.columns:
    compare.string("display_airport_city_name_full", "display_airport_city_name_full", method="jarowinkler", label="City_Sim")
if "display_airport_name" in stations.columns:
    compare.string("display_airport_name", "display_airport_name", method="jarowinkler", label="Name_Sim")

features = compare.compute(candidate_links, stations)
threshold = 1.75 if len(features.columns) == 2 else 0.85
matches = features[features.sum(axis=1) > threshold]

print(f"Identified {len(matches)} near-duplicate candidates.")

if not matches.empty:
    i1 = matches.index.get_level_values(0)
    i2 = matches.index.get_level_values(1)

    dup_view = pd.DataFrame(
        {
            "Code_1": stations.loc[i1, "airport"].values,
            "Code_2": stations.loc[i2, "airport"].values,
            "City_1": stations.loc[i1, "display_airport_city_name_full"].values if "display_airport_city_name_full" in stations.columns else "N/A",
            "City_2": stations.loc[i2, "display_airport_city_name_full"].values if "display_airport_city_name_full" in stations.columns else "N/A",
            "Name_1": stations.loc[i1, "display_airport_name"].values if "display_airport_name" in stations.columns else "N/A",
            "Name_2": stations.loc[i2, "display_airport_name"].values if "display_airport_name" in stations.columns else "N/A",
            "Score": matches.sum(axis=1).values,
        }
    ).sort_values("Score", ascending=False)

    out_dup = CLEANING_REPORT_DIR / "Near_Duplicates_Station.csv"
    dup_view.to_csv(out_dup, index=False)
    print(f"Near-duplicates list saved -> {out_dup}")

## Cleaning Summary Report

In [ ]:
summary_rows = []

for table_name in dfs_raw:
    raw = dfs_raw[table_name]
    clean = dfs_clean.get(table_name, raw)
    summary_rows.append(
        {
            "Table": table_name,
            "Rows_Before": raw_row_counts.get(table_name, len(raw)),
            "Rows_After": len(clean),
            "Rows_Removed": raw_row_counts.get(table_name, len(raw)) - len(clean),
            "Cols": len(clean.columns),
            "Nulls_Before": raw_null_counts.get(table_name, int(raw.isnull().sum().sum())),
            "Nulls_After": int(clean.isnull().sum().sum()),
        }
    )

summary_df = pd.DataFrame(summary_rows)
print("\n--- Final Cleaning Summary ---")
print(summary_df.to_string(index=False))

out_sum = CLEANING_REPORT_DIR / "Cleaning_Summary.csv"
summary_df.to_csv(out_sum, index=False)
print(f"Summary saved -> {out_sum}")

## Post-Cleaning Schema Validation

In [ ]:
# Validate cleaned tables against original schemas (strict=False allows column additions)
report = contract.validate_all(dfs_clean, strict=False)
schema_validation_rows = []
for table_name, result in report["results"].items():
    for detail in result["details"]:
        schema_validation_rows.append({"Table": table_name, "Overall_Status": result["status"], **detail})
pd.DataFrame(schema_validation_rows).to_csv(CLEANING_REPORT_DIR / "Post_Cleaning_Schema_Validation.csv", index=False)
print(f"Schema validation details saved -> {CLEANING_REPORT_DIR / 'Post_Cleaning_Schema_Validation.csv'}")
if report["failures"] == 0:
    print("Schema validation PASSED - all cleaned tables match expected schemas.")
else:
    print("Schema validation FAILED - structural changes detected!")
    for entry in report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")